[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C00_VLM_Multimodal_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检（环境医生 / Environment Doctor）

这是一份**纯 CPU、不下载任何模型、不联网**的环境体检 notebook。从上到下跑一遍，确认你的 `vlm` 环境已经配好，再进模块 01。

它会逐项检查：
1. **关键库版本** —— `torch / transformers / accelerate / peft / datasets`（逐个 `try/except` import，缺的报红）
2. **加速设备** —— CUDA 是否可用、设备名、显存（GB）；Apple `mps`；给出**推荐 device**
3. **量化后端** —— `bitsandbytes`（仅 CUDA 有意义），缺失给提示
4. **API key 是否设置** —— 只打印 `bool`，**绝不打印 key 的值**
5. **轻量 smoke test** —— 在推荐 device 上做一次小矩阵乘 + 简单 autograd，验证算子链路正常
6. **每模块算力需求表** —— 与 README 一致
7. **3 道 ✏️ 练习** —— 版本号比较、显存估算、优雅回退 import，巩固本模块的环境常识

> 看到 `[ OK ]` 就是绿灯，`[MISS]` 表示该库缺失或环境变量未设置。`bitsandbytes` 在 CPU/MPS 机器上缺失是正常的。

In [1]:
# ── Cell 1：基础环境与关键库版本 ──
# 逐个 try/except import，缺哪个一目了然；不会因为一个库缺失就让整个 cell 崩掉。
import sys, platform

print(f"Python      : {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")
print("-" * 56)

def check_import(name, attr="__version__"):
    try:
        mod = __import__(name)
        ver = getattr(mod, attr, "?")
        print(f"[ OK ]  {name:<14} {ver}")
        return mod
    except Exception as e:
        print(f"[MISS]  {name:<14} 未安装或导入失败 -> {type(e).__name__}: {e}")
        return None

torch        = check_import("torch")
transformers = check_import("transformers")
accelerate   = check_import("accelerate")
peft         = check_import("peft")
datasets     = check_import("datasets")

print("-" * 56)
if torch is None:
    print("⚠️  torch 缺失！请先按 PyTorch 官网命令安装对应 CUDA 版本，再 pip install -r requirements.txt")
else:
    print("✅ 核心库检查完毕。缺失的项按 requirements.txt 补装即可。")

Python      : 3.12.13  (Linux x86_64)
--------------------------------------------------------
[ OK ]  torch          2.11.0+cu128
[ OK ]  transformers   5.13.1
[ OK ]  accelerate     1.14.0
[ OK ]  peft           0.19.1
[ OK ]  datasets       4.0.0
--------------------------------------------------------
✅ 核心库检查完毕。缺失的项按 requirements.txt 补装即可。


## 检测加速设备：CUDA / MPS → 推荐 device

下面探测可用的加速后端：
- **CUDA**（NVIDIA 显卡）：打印设备名与总显存（GB），多卡会逐个列出。
- **MPS**（Apple Silicon 的 Metal 后端）：macOS 上可用。
- 最后给出一个**推荐 `device` 字符串**，这正是各模块 notebook 首个 code cell 选 device 的同款逻辑：
  ```python
  device = "cuda" if cuda else ("mps" if mps else "cpu")
  ```

In [2]:
# ── Cell 2：检测 CUDA / MPS，给出推荐 device ──
if torch is None:
    print("torch 未安装，跳过设备检测。")
    device = "cpu"
else:
    cuda_ok = torch.cuda.is_available()
    mps_ok  = getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available()

    print(f"CUDA available : {cuda_ok}")
    if cuda_ok:
        n = torch.cuda.device_count()
        print(f"  CUDA devices : {n}")
        for i in range(n):
            name = torch.cuda.get_device_name(i)
            total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
            print(f"    [{i}] {name}  |  {total_gb:.1f} GB")
        print(f"  CUDA runtime : {torch.version.cuda}")

    print(f"MPS  available : {mps_ok}  (Apple Silicon Metal 后端)")

    # 推荐 device：与各模块 notebook 首个 cell 一致的选择逻辑
    device = "cuda" if cuda_ok else ("mps" if mps_ok else "cpu")
    print("-" * 56)
    print(f"👉 推荐 device = \"{device}\"")
    if device == "cpu":
        print("   无 GPU/MPS：CPU 可跑 CPU 标注的实验；GPU 标注的重型 cell 建议用 Colab/云卡（见模块 00 第 5 节）。")

CUDA available : True
  CUDA devices : 1
    [0] Tesla T4  |  14.6 GB
  CUDA runtime : 12.8
MPS  available : False  (Apple Silicon Metal 后端)
--------------------------------------------------------
👉 推荐 device = "cuda"


## 检测量化后端：bitsandbytes（仅 CUDA 有意义）

`bitsandbytes` 提供 4/8-bit 量化，是本教材在单张 16–24GB 卡上跑 7B 级模型 / QLoRA 微调的关键。

注意：**它基本只在 Linux + CUDA 上工作**。在 CPU 或 macOS/MPS 机器上缺失是**完全正常**的 —— 那些机器本来也不会用它做 4-bit 推理。

In [3]:
# ── Cell 3：检测 bitsandbytes ──
try:
    import bitsandbytes as bnb
    print(f"[ OK ]  bitsandbytes  {bnb.__version__}")
    if torch is not None and not torch.cuda.is_available():
        print("   ⚠️ 已安装，但当前无 CUDA —— bitsandbytes 的 4/8-bit 量化需要 NVIDIA GPU 才能生效。")
except Exception as e:
    print(f"[MISS]  bitsandbytes 未安装/不可用 -> {type(e).__name__}")
    if torch is not None and torch.cuda.is_available():
        print("   你有 CUDA，建议安装以启用 4-bit 量化：pip install bitsandbytes")
    else:
        print("   当前是 CPU/MPS 环境，缺失属正常 —— 量化推理需要 NVIDIA GPU，可暂时忽略。")

[MISS]  bitsandbytes 未安装/不可用 -> ModuleNotFoundError
   你有 CUDA，建议安装以启用 4-bit 量化：pip install bitsandbytes


In [5]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00:00:0100:01


## 检测 API key 是否设置（只看存在与否，绝不打印值）

模块 07/08/09 用闭源模型做评测对比时需要这些 key。这里**只打印布尔值**（是否设置），
**绝不打印 key 的内容** —— 这是处理凭据的基本卫生，避免 key 出现在 notebook 输出里被误提交。

未设置不影响纯开源模块；用到时再 `export` 即可。

In [6]:
# ── Cell 4：检测环境变量是否设置（只打印 bool）──
import os

KEYS = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "HF_TOKEN"]
print("API key / token 是否已设置（只显示 True/False，不显示内容）：")
print("-" * 56)
for k in KEYS:
    is_set = bool(os.environ.get(k))
    flag = "[ SET ]" if is_set else "[ --- ]"
    print(f"{flag}  {k:<20} {is_set}")
print("-" * 56)
print("未设置不影响纯开源模块；07/08/09 用到闭源模型对比时再 export 即可。")

API key / token 是否已设置（只显示 True/False，不显示内容）：
--------------------------------------------------------
[ --- ]  OPENAI_API_KEY       False
[ --- ]  ANTHROPIC_API_KEY    False
[ --- ]  HF_TOKEN             False
--------------------------------------------------------
未设置不影响纯开源模块；07/08/09 用到闭源模型对比时再 export 即可。


## 轻量 smoke test：小矩阵乘 + 简单 autograd

最后做一个**极轻量**的算子链路体检：在推荐 `device` 上建两个小张量，做矩阵乘，再走一次反向传播。
**不下载任何模型、不联网**，几毫秒就能跑完。目的只有一个：确认 `torch` 在这台机器上前向 / 反向 / 设备搬运都正常。

看输出：
- `matmul` 结果形状应为 `(4, 4)`；
- `x.grad` 不为 `None` 且形状与 `x` 一致 → autograd 链路通畅。

In [7]:
# ── Cell 5：smoke test（不下载任何模型）──
if torch is None:
    print("torch 未安装，跳过 smoke test。")
else:
    print(f"在 device = {device} 上运行 smoke test ...")

    # 前向：小矩阵乘
    a = torch.randn(4, 8, device=device)
    b = torch.randn(8, 4, device=device)
    c = a @ b
    assert c.shape == (4, 4), "matmul 形状不对！"
    print(f"  matmul 输出形状: {tuple(c.shape)}  ->  OK")

    # 反向：简单 autograd
    x = torch.randn(4, 4, device=device, requires_grad=True)
    loss = (x ** 2).sum()       # 标量 loss，d(loss)/dx = 2x
    loss.backward()
    assert x.grad is not None and x.grad.shape == x.shape, "autograd 失败！"
    max_err = (x.grad - 2 * x).abs().max().item()
    print(f"  autograd: x.grad 形状 {tuple(x.grad.shape)}，与解析解 2x 的最大误差 = {max_err:.2e}  ->  OK")

    print("-" * 56)
    print("✅ smoke test 通过：前向 / 反向 / 设备搬运均正常。")

在 device = cuda 上运行 smoke test ...
  matmul 输出形状: (4, 4)  ->  OK
  autograd: x.grad 形状 (4, 4)，与解析解 2x 的最大误差 = 0.00e+00  ->  OK
--------------------------------------------------------
✅ smoke test 通过：前向 / 反向 / 设备搬运均正常。


## 每模块算力需求表

下表与 README / 主页一致，方便你按手头硬件规划学习顺序。优先安装 `pandas` 时用 DataFrame 展示，否则退化为纯文本表格 —— 两种都能看。

- **CPU**：笔记本即可跑
- **GPU**：**Colab 免费 T4（16GB）就够用**——课程里的模型多是 2–7B 级，fp16 直接放得下；只有个别把多个模型叠着用的 cell 需要在加载下一个前 `del` 释放显存，不需要默认量化
- **API**：需要闭源模型 key 做对比 / 当 judge

In [ ]:
# ── Cell 6：打印每模块算力需求表 ──
rows = [
    ("00", "环境与总览",                 "CPU"),
    ("01", "多模态基础与视觉编码器",       "CPU / GPU"),
    ("02", "对比对齐：CLIP / SigLIP",      "GPU"),
    ("03", "VLM 架构演进",                "GPU"),
    ("04", "连接器与训练范式",             "GPU"),
    ("05", "视觉指令微调与对齐",           "GPU"),
    ("06", "高分辨率 / 任意分辨率 / 视频",  "GPU"),
    ("07", "VLM 评测体系 ★",              "GPU / API"),
    ("08", "幻觉、鲁棒性与安全评估 ★",      "GPU / API"),
    ("09", "原生多模态与多模态智能体",      "GPU / API"),
]

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["#", "模块", "算力"])
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string(index=False))
except Exception:
    # 无 pandas 时的纯文本回退
    print(f"{'#':<4}{'模块':<26}{'算力'}")
    print("-" * 56)
    for n, name, comp in rows:
        print(f"{n:<4}{name:<26}{comp}")

---
## ✏️ 练习 1：实现 `version_at_least`

Cell 1 打印了各库的版本字符串，但「够不够新」还得人眼判断。实现 `version_at_least(ver, minimum)`：输入两个版本字符串（如 `"2.1.0+cu121"` 和 `"2.0"`），当 `ver >= minimum` 时返回 `True`。

**提示**：先把 `"+"` 之后的 local 段切掉（PyTorch 常见的 `+cu121` 不参与比较）；再按 `"."` split 并**转 int** 得到数字元组——按字符串比会得出 `"2.10" < "2.9"` 的错误结论；两元组用 0 补齐到等长（`"2.1"` 等价于 `"2.1.0"`）后直接比较，Python 的元组比较天然就是逐位的。12 行以内。

In [ ]:
def version_at_least(ver, minimum):
    # TODO: 1) 去掉 "+" 后的 local 段（"2.1.0+cu121" -> "2.1.0"）
    #       2) 按 "." split 并转 int，得到数字元组
    #       3) 两元组用 0 补齐到等长，返回逐位比较结果 ver >= minimum
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert version_at_least("2.1.0+cu121", "2.0")          # local 段 +cu121 应被忽略
assert version_at_least("2.10.0", "2.9.1")             # 数值比较：10 > 9（字符串比较会出错）
assert not version_at_least("4.38.2", "4.40.0")
assert version_at_least("2.1", "2.1.0")                # 边界：位数不同，0 补齐后相等
assert not version_at_least("0.9.9", "1.0")
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `estimate_weight_memory_gb`

Cell 2 打印了显卡总显存，但「7B 模型放不放得下」还要会算。实现 `estimate_weight_memory_gb(n_params, dtype)`：给定参数量与精度，返回**权重本身**占用的显存（GB）。

公式：显存（GB）= 参数量 × 每参数字节数 / 1024³。每参数字节数：`fp32`=4、`fp16`/`bf16`=2、`int8`=1、`int4`=0.5；遇到不认识的 dtype 抛 `ValueError`。

**提示**：一个 dict 映射 + 一行算式，8 行以内。算完体会一下：7B 模型 fp16 权重 ≈ 13 GB——**Colab 的 T4（16GB）刚好放得下**，KV cache 和激活还要另算，所以显存紧张时通常先减 batch/分辨率而不是急着上量化；`int4`/`int8` 这两列留着是给你对比压缩比，本课默认不会因为这几 GB 的差距就默认量化。

In [ ]:
def estimate_weight_memory_gb(n_params, dtype):
    # TODO: 1) bytes_per_param 映射表：fp32=4, fp16/bf16=2, int8=1, int4=0.5
    #       2) dtype 不在表内 -> raise ValueError
    #       3) 返回 n_params * 每参数字节数 / 1024**3
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(estimate_weight_memory_gb(7e9, "fp16") - 13.04) < 0.01    # 7B fp16 ≈ 13 GB
assert abs(estimate_weight_memory_gb(1e9, "fp32") - 3.73) < 0.01     # 1B fp32 ≈ 3.7 GB
assert estimate_weight_memory_gb(7e9, "bf16") == estimate_weight_memory_gb(7e9, "fp16")
assert estimate_weight_memory_gb(7e9, "fp16") / estimate_weight_memory_gb(7e9, "int4") == 4.0  # 4-bit 压缩 4 倍
try:
    estimate_weight_memory_gb(7e9, "fp8")
    assert False, "未知 dtype 应抛 ValueError"
except ValueError:
    pass
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `check_import_ex`

不翻上文，自己实现一遍 Cell 1 的优雅回退 import：`check_import_ex(name, attr="__version__")`——导入成功返回该属性的值（模块没有该属性时返回 `"?"`），导入失败返回 `None`。**无论传入什么，函数本身绝不抛异常**——这正是环境自检脚本能「逐项报告、一项缺失不挂全局」的关键。

**提示**：`__import__(name)` 接受字符串模块名；`getattr(mod, attr, "?")` 自带默认值；整体套一层 `try/except Exception` 兜底。6 行以内。

In [ ]:
def check_import_ex(name, attr="__version__"):
    # TODO: try 导入 name；成功 -> 返回 getattr(模块, attr, "?")；任何异常 -> 返回 None
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
import sys
assert check_import_ex("json") is not None                       # 标准库，导入必成功
assert check_import_ex("math") == "?"                            # math 没有 __version__ -> 回退 "?"
assert check_import_ex("sys", attr="platform") == sys.platform   # attr 可换：读任意模块属性
assert check_import_ex("definitely_no_such_module_42") is None   # 不存在的模块 -> None，而非抛异常
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def version_at_least(ver, minimum):
    def to_tuple(s):
        s = s.split("+")[0]                       # 去掉 +cu121 这类 local 段
        return tuple(int(x) for x in s.split("."))
    a, b = to_tuple(ver), to_tuple(minimum)
    n = max(len(a), len(b))
    a += (0,) * (n - len(a))                      # 位数不同，0 补齐
    b += (0,) * (n - len(b))
    return a >= b                                 # 元组比较天然逐位

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def estimate_weight_memory_gb(n_params, dtype):
    bytes_per_param = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5}
    if dtype not in bytes_per_param:
        raise ValueError(f"未知 dtype: {dtype!r}，支持 {sorted(bytes_per_param)}")
    return n_params * bytes_per_param[dtype] / 1024 ** 3

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def check_import_ex(name, attr="__version__"):
    try:
        mod = __import__(name)
        return getattr(mod, attr, "?")
    except Exception:
        return None

## 小结

如果上面 6 个 cell 都跑出绿灯（关键库 `[ OK ]`、推荐 device 已给出、smoke test 通过），你的 `vlm` 环境就绪了。

常见情况：
- `bitsandbytes` 在 **CPU/macOS** 上 `[MISS]` —— 正常，量化推理需要 NVIDIA GPU。
- API key 全是 `[ --- ]` —— 正常，到 07/08/09 用闭源对比时再 `export`。
- 某个核心库 `[MISS]` —— 回模块 00 的「环境配置详解」，先单独装好 `torch`，再 `pip install -r requirements.txt`。

**下一步 → 模块 01 · 多模态基础与视觉编码器**：从 ViT 出发，看图像如何变成 token 序列。
打开 `01_foundations/01_讲解.html` 读讲解，再跑 `01_foundations/01_vision_encoders.ipynb`。